### NOMEANDO VARIÁVEIS

In [0]:
from pyspark.sql import functions as F      # Funções spark como F.when() ou "functions.when()"
from pyspark.sql.window import Window       # Biblioteca necessária pra tratamento de duplicados mantendo o mais recente

# Guardando novamenteo nome do catalog
catalog = "workspace"

# Usando os mesmos nomes do Landing_to_Bronze para as três databases (schemas) 
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

# Novamente guardando o caminho completo de cada schema (catalog + nome do schema).
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Print pra validar
print(f"silver_schema: {silver_schema}")

silver_schema: workspace.silver


### CRIANDO FUNÇÕES GERAIS

In [0]:
# FUNÇÕES DE CHECAGEM - BASEADAS NAS FUNÇÕES DO NOTEBOOK MOSTRADO EM AULA

from datetime import datetime

# Lista que vai guardar o resultado de cada checagem de qualidade feita no notebook
dq_results = []

# df.count() - Conta quantas linhas tem o dataframe
# df.filter(~condition).count() - Filtra as linhas que não satisfazem a condition passada e conta quantas são
# dq_results.append() - Adiciona o resultado da checagem na lista dq_results

# Conta quantas linhas violam a condição esperada e guarda o resultado
def dq_check(table_name, check_name, df, condition):
    total = df.count()      
    failed = df.filter(~condition).count()

    # Verificação
    passed = failed == 0    

    dq_results.append(
        {"table_name": table_name, "check_name": check_name, "total_rows": total,
         "failed_rows": failed, "passed": passed, "checked_at": datetime.now()}
    )
    
    # Atribui o resultado da checagem na coluna status
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")


# df.count() - Conta quantas linhas tem o dataframe
# df.groupBy(*key_cols) - O * "desempacota" a lista, se key_cols = ["id_filme"], ele vai contar quantas linhas tem o msm ID ("primary key")
# .filter("count > 1") - Mantém só os grupos que aparecem mais de uma vez

# Checagem específica: verifica se as colunas-chave são únicas (sem duplicatas)
def dq_check_unique(table_name, check_name, df, key_cols):
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    
    # Verificação
    passed = dupes == 0
    
    dq_results.append(
        {"table_name": table_name, "check_name": check_name, "total_rows": total,
         "failed_rows": dupes, "passed": passed, "checked_at": datetime.now()}
    )

    # Atribui o resultado da checagem na coluna status
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

# print pra confirmar
print("Funções definidas com sucesso!")

Funções definidas com sucesso!


In [0]:
# FUNÇÃO PRA RESOLVER DEDUPLICAÇÃO

# Tratamento de dados duplicados é na silver, nao na bronze (bronze é append only)
# USANDO AQUELE window q eu importei do "pyspark.sql"

# Window.partitionBy() - divide os dados em partições separados pelo id (agrupa linhas com id igual)
# .orderBy() - função de ordenar padrao
# F.desc() - Ordenado de forma decrescente de data de inserção

# Remove duplicatas mantendo a linha mais recente por chave, baseado numa coluna de data.
def deduplicar_por_chave(df, colunas_chave, coluna_data="ingestion_datetime"):
    janela = Window.partitionBy(*colunas_chave).orderBy(F.desc(coluna_data))

    df_dedup = (
        df
        .withColumn("linha_numero", F.row_number().over(janela))
        .filter(F.col("linha_numero") == 1)
        .drop("linha_numero")
    )

    print(f"Antes da deduplicação: {df.count()} linhas")
    print(f"Depois da deduplicação: {df_dedup.count()} linhas")

    return df_dedup

# Print de confirmação
print("Função de deduplicação definida com sucesso!")

Função de deduplicação definida com sucesso!


### CRIANDO DATABASE SE NAO EXISTIR

In [0]:
# CRIANDO DATABASE SE NAO EXISTIR

# Cria o database (schema) silver se ele nao existir
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_schema}")

print(f"silver_schema: {silver_schema} criado/confirmado com sucesso.")

silver_schema: workspace.silver criado/confirmado com sucesso.


### TABELA `silver.tb_movies_info`

In [0]:
# LENDO A TABELA BRONZE MOVIES_INFO E VENDO OS DIFERENTES STATUS

# Salvando a tabela bronze em uma variável pra poder usar aq na silver
df_bronze_movies_info = spark.table(f"{bronze_schema}.tb_movies_info")

# Como a coluna status precisa ser normalizada(remover sujeira) antes de ser traduzida pra portugues
# Da um select em status que são diferentes e truncando
#       distinct() - meio auto explicativo, mas é pra retornar somente os diferentes
#       show(truncate=False) - mostra o resultado truncado pra ficar apenas um por linha
df_bronze_movies_info.select("status").distinct().show(truncate=False)

+---------------------------------------------------------------------------+
|status                                                                     |
+---------------------------------------------------------------------------+
|In-Production                                                              |
|in production                                                              |
| sometimes referred to the Islamic Religious Police. All this              |
|POST PRODUCTION                                                            |
|Planned                                                                    |
|NULL                                                                       |
|post production                                                            |
|released                                                                   |
|\Can you still open your heart to a friend who turns out to be ""\""Boy A""|
| his errant dad returns                                        

In [0]:
# NORMALIZAÇÃO E TRADUÇÃO DA COLUNA STATUS DO tb_movies_info

# Criando uma variável "df_status_traduzido" pra receber os dados normalizados e traduzidos

# APOS A MONITORIA DECIDI QUEBRAR 
#           ".withColumn("status", F.upper(F.trim(F.regexp_replace(F.col("status"), "-", " "))))" 
# EM LINHAS DIFERENTES PARA FICAR COM O CODIGO MAIS LEGÍVEL

df_status_traduzido = (
    # Removendo hifen, espaço e ajustando para caixa alta
    df_bronze_movies_info
    .withColumn("status_sem_hifen", F.regexp_replace(F.col("status"), "-", " "))    # Troca hífen por espaço
    .withColumn("status_sem_espacos", F.trim(F.col("status_sem_hifen")))    # Remove espaços sobrando no início/fim
    .withColumn("status", F.upper(F.col("status_sem_espacos")))     # Deixa tudo em maiúsculas
    
    # Traduz o status já limpo
    .withColumn("status_filme",
        F.when(F.col("status") == "RELEASED", "Lançado")
        .when(F.col("status") == "POST PRODUCTION", "Pós-Produção")
        .when(F.col("status") == "IN PRODUCTION", "Em Produção")
        .when(F.col("status") == "PLANNED", "Planejado")
        .when(F.col("status") == "RUMORED", "Rumores")
        .when(F.col("status") == "CANCELED", "Cancelado")
        .otherwise("Não Informado")
    )

    # Remoção das colunas auxs que criei pra facilitar o processo de normalização
    .drop("status_sem_hifen", "status_sem_espacos")
)

# Print pra confirmação 
print("Contagem por status traduzido:")
df_status_traduzido.groupBy("status_filme").count().orderBy(F.desc("count")).show(truncate=False)

Contagem por status traduzido:
+-------------+------+
|status_filme |count |
+-------------+------+
|Lançado      |527015|
|Pós-Produção |3715  |
|Em Produção  |3330  |
|Não Informado|350   |
|Planejado    |240   |
+-------------+------+



In [0]:
# RESOLVENDO A DEDUPLICAÇÃO

df_deduplicado = deduplicar_por_chave(df_status_traduzido, ["id"])

print("\nContagem por status traduzido:")
df_deduplicado.groupBy("status_filme").count().orderBy(F.desc("count")).show(truncate=False)

# OBS: a 2ª camada de dedup (mesmo filme com "id" diferente) foi movida para depois da
# conversão de data, porque aqui "release_date" ainda é STRING em 3 formatos diferentes
# (ISO/US/BR) -- o mesmo filme com a data escrita em formatos diferentes escapava da dedup.

Antes da deduplicação: 534650 linhas
Depois da deduplicação: 97879 linhas

Contagem por status traduzido:
+-------------+-----+
|status_filme |count|
+-------------+-----+
|Lançado      |96463|
|Pós-Produção |701  |
|Em Produção  |604  |
|Não Informado|64   |
|Planejado    |47   |
+-------------+-----+



In [0]:
# VERIFICANDO OS DIFERENTES FORMATOS DE DATAS NO CAMPO DE DATAS DE LANÇAMENTO

# esqueci que cada data é uma data kkkkkkkkk, mas vou deixar como símbolo de linhas de raciocínio
print("Formatos de datas de lançamento:")
df_deduplicado.select("release_date").distinct().show(truncate=False)

Formatos de datas de lançamento:
+------------+
|release_date|
+------------+
|2017-11-13  |
|2019-10-23  |
|2016-08-12  |
|2016-04-18  |
|2017-02-08  |
|2016-01-13  |
|2017-01-06  |
|2016-12-08  |
|2016-04-05  |
|2017-02-17  |
|2016-05-13  |
|10-13-2016  |
|2016-08-10  |
|2016-05-27  |
|2016-01-22  |
|2016-01-15  |
|2017-06-22  |
|2017-12-01  |
|2017-01-07  |
|2018-05-11  |
+------------+
only showing top 20 rows


In [0]:
# CONVERSÃO DE DATA MULTI FORMATO

# como foi aferido, as datas podem ter 3 formatos iso(yyyy-MM-dd), us(MM-dd-yyyy) e bra(dd/MM/yyyy). A ideia é unificar tudo numa só notação

# ========= FUNCIONAMENTO ===========
# Usar try_to_date() pq é uma forma de converter um valor python pra uma data que o spark entende
# Em "df_com_data_convertida" tenta-se converter a data usando os formatos iso, us e br, na ordem 
# Se uma conversão der certo, ele usa aquele resultado. Se todas derem errado, ele retorna NULL
# Sendo uma data válida, uma das 3 colunas não será NULL
# Depois passa linha por linha selecionando aquela data que nao for NULL e descartando o resto

# Criando dataframe com a data ja convertida    
df_com_data_convertida = (
    df_deduplicado

    # F.try_to_date(coluna, formato) - tenta interpretar o texto da coluna como se fosse uma data escrita naquele formato específico
    #   F.col() - A coluna que vai ser convertida
    #   F.lit() - O formato que vai ser usado na conversão
    .withColumn("data_formato_iso", F.try_to_date(F.col("release_date"), F.lit("yyyy-MM-dd")))
    .withColumn("data_formato_us", F.try_to_date(F.col("release_date"), F.lit("MM-dd-yyyy")))
    .withColumn("data_formato_br", F.try_to_date(F.col("release_date"), F.lit("dd/MM/yyyy")))

    # F.coalesce(coluna 1, coluna 2, coluna 3) - Testa coluna a coluna e o primeiro valos diferente de NULL encontrado
    .withColumn("data_lancamento",
        F.coalesce(F.col("data_formato_iso"), F.col("data_formato_us"), F.col("data_formato_br"))
    )
    .drop("data_formato_iso", "data_formato_us", "data_formato_br")     # limpeza
)

# Variáveis contadoras pra confirmação
total_linhas = df_com_data_convertida.count()
datas_nulas = df_com_data_convertida.filter(F.col("data_lancamento").isNull()).count()

# Mostrando resultados
print(f"Total de linhas: {total_linhas}")
print(f"Datas que são NULL (nao se encaixaram nos formatos): {datas_nulas}")

display(df_com_data_convertida.select("release_date", "data_lancamento").limit(50))

# O FORMATO DE DATA FICOU YYYY-MM-DD PQ É O FORMATO DA VARIÁVEL DATE DO DATABRICKS (resultado do try_to_date()), pelo menos pelo oq eu pesquisei é isso

Total de linhas: 97879
Datas que são NULL (nao se encaixaram nos formatos): 64


release_date,data_lancamento
2017-02-01,2017-02-01
2020-02-17,2020-02-17
2022-06-22,2022-06-22
2016-07-14,2016-07-14
2018-01-01,2018-01-01
23/05/2022,2022-05-23
26/09/2017,2017-09-26
2021-01-21,2021-01-21
2016-09-14,2016-09-14
2016-09-09,2016-09-09


In [0]:
# CRIANDO COLUNA ANO LANÇAMENTO DERIVADA DA DATA DE LANÇAMENTO

# F.year(coluna) - pega so o ano de uma coluna tipo date
# Se a data de lançamento for NULL, o ano de lançamento tbm é automaticamente NULL
df_com_ano = df_com_data_convertida.withColumn(
    "ano_lancamento", F.year(F.col("data_lancamento"))
)

# Verificação
display(df_com_ano.select("data_lancamento", "ano_lancamento").limit(10))

data_lancamento,ano_lancamento
2017-02-01,2017
2020-02-17,2020
2022-06-22,2022
2016-07-14,2016
2018-01-01,2018
2022-05-23,2022
2017-09-26,2017
2021-01-21,2021
2016-09-14,2016
2016-09-09,2016


In [0]:
# Verificando somente os nulos agr
display(
    df_com_ano
    .filter(F.col("data_lancamento").isNull())
    .select("release_date", "data_lancamento", "ano_lancamento")
    .limit(20)
)

# OS RUÍDOS PRESENTES NO DATARELEASE VIRARAM NULLS DE FATO, tudo certo

release_date,data_lancamento,ano_lancamento
null,null,null
null,null,null
null,null,null
World's greatest detective. World's smallest package.,null,null
null,null,null
null,null,null
null,null,null
null,null,null
null,null,null
null,null,null


In [0]:
# PADRONIZANDO O NOME DAS COLUNAS PARA O PADRÃO SILVER ESTABELECIDO NO DOCUMENTO .png

# SEGUNDA CAMADA DE DEDUPLICAÇÃO: a dedup da célula anterior só pega duplicata EXATA de "id"
# Só que a base tem o MESMO FILME cadastrado várias vezes com "id" diferentes (mesmo título + mesma data de lançamento) descoberto ao validar a Pergunta 5 da Gold
# Rodamos AQUI (e não lá em cima) porque só agora "data_lancamento" é uma DATE de verdade
# Antes, "release_date" era string em 3 formatos (ISO/US/BR), então o mesmo filme com a data escrita em formatos diferentes escapava da dedup e voltava a aparecer duplicado na Gold
janela_duplicata_filme = Window.partitionBy(
    F.lower(F.trim(F.col("title"))), F.col("data_lancamento")
).orderBy(F.col("id"))

qtd_antes_filme_dup = df_com_ano.count()

df_com_ano_dedup = (
    df_com_ano
    .withColumn("_rn_duplicata_filme", F.row_number().over(janela_duplicata_filme))
    .filter(F.col("_rn_duplicata_filme") == 1)
    .drop("_rn_duplicata_filme")
)

print(f"Antes da 2ª dedup (mesmo filme, id diferente): {qtd_antes_filme_dup} linhas")
print(f"Depois da 2ª dedup: {df_com_ano_dedup.count()} linhas")

df_silver_info_filmes = (

    # .withColumnRenamed("nome_antigo", "nome_novo") - Renomear uma coluna já existente
    #  As variáveis que já estão com o nome certo logicamente n precisam ser renomeadas
    df_com_ano_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    # Select somente as colunas que vão ser usadas na tabela final
    .select(
        "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
        "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao"
    )
)

# Checagens de qualidade (por condição e por duplicata)
dq_check_unique("silver.tb_info_filmes", "id_filme único", df_silver_info_filmes, ["id_filme"])
dq_check("silver.tb_info_filmes", "id_filme não nulo", df_silver_info_filmes, F.col("id_filme").isNotNull())

# Gravando da tabela Silver (dessa vez com modo overwrite pq nao é na bronze append only)
df_silver_info_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

# Print de confirmação
print("Tabela silver.tb_info_filmes gravada com sucesso.")
display(df_silver_info_filmes.limit(100))

Antes da 2ª dedup (mesmo filme, id diferente): 97879 linhas
Depois da 2ª dedup: 97468 linhas
[PASS] silver.tb_info_filmes | id_filme único | 0 chaves duplicadas de 97468 linhas
[PASS] silver.tb_info_filmes | id_filme não nulo | 0/97468 linhas falharam
Tabela silver.tb_info_filmes gravada com sucesso.


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
564096,"""\""""\""""BLESSED""""\""""\""""""""""\""""\""""BLESSED""""\""""\""""""""en22/11/201885ReleasedSupreme presents"" \""""\""""BLESSED\""""\"""" a full length video directed by William Strobeck featuring Tyshawn Jones""""""",null,null,null,null,null,Não Informado,null,null
1144991,#69 Samskar Colony,#69 Samskar Colony,2022-03-18,2022,116,te,Lançado,"Koushik, a teenage boy, moves to the city with his family as a tenant in the Samskar colony where he falls in love with the landlady Vaishali, a married woman.",null
584871,#AbroHilo,#AbroHilo,2019-02-23,2019,54,es,Lançado,"Humor shapes the way Spaniards interact on Twitter: all sorts of topics can be used to make a joke and many anonymous commentators can become celebrities and compete with professional comedians. But sometimes certain jokes that defy political correctness have a high price for those who dare to make them, jokes that can freeze the smiles of thousands of people whose prejudices can put an end to some very successful artistic careers.",Humor on Twitter told by its protagonists
614696,#Alive,#살아있다,2020-06-24,2020,98,ko,Lançado,"As a grisly virus rampages a city, a lone man stays locked inside his apartment, digitally cut off from seeking help and desperate to find a way out.",You must survive.
476424,#Captured,#Captured,2017-09-05,2017,81,en,Lançado,A zealous vigilante looking to clean the Internet of sin targets a group of teens that broadcast misadventures on their chatroom.,Cleansing the internet of all sin
1036134,#Clout,#Clout,2022-10-11,2022,140,en,Lançado,Four stories unfold in Atlanta around the city's top influencers and their dependence on social media and their thirst for online prominence.,What would you do for it?
751146,#FollowMe,#FollowMe,2020-10-07,2020,22,en,Lançado,A documentary that enters into the world of Iraqi social media influencers and follows their perilous journey as they fight for their rights.,null
503352,#FriendButMarried,#TemanTapiMenikah,2018-03-28,2018,102,id,Lançado,"Pining for his high school crush for years, a young man puts up his best efforts to move out of the friend zone until she reveals she's getting married.",null
655293,#FriendButMarried 2,#TemanTapiMenikah 2,2020-02-27,2020,104,id,Lançado,"As Ayu and Ditto finally transition from best friends to newlyweds, a quick pregnancy creates uncertainty for the future of their young marriage.",null
716733,#HandballStrive,#ハンド全力,2020-07-24,2020,108,ja,Lançado,"The school’s handball club is about to close down. Can social media bring it back to life? Masao Kiyota is a high school student living in Japan’s southern Kumamoto Prefecture. Lacking passion for anything in life, he spends his days like so many youth on his smartphone along with his childhood friend, Okamoto. One day, they upload a photo taken three years previously when they were both part of their school’s handball team. To their surprise, the post goes a little viral. Encouraged, they add the hashtag: “#Handball Full Power” and are swarmed with “likes” from around the country. Amidst the sudden social media attention, Masao and Okamoto set to resuscitating a nearly defunct men’s handball team.",null


In [0]:
# CORRIGINDO SUJEIRA RESIDUAL NO TITULO 

# PASSEI UM TEMPAO TENTANDO ARRUMAR OS FILTROS PRA NAO FICAR COM NENHUMA SUJEIRA NOS TÍTULOS, MAS ISSO SEMPRE ME FAZIA PERDER ALGUNS TÍTULOS VÁLIDOS
# COMO TÍTULO É ALGO MT LIVRE DE SE ESCOLHER, FICA DE FATO MT DIFICIL DE FILTRAR A SUJEIRA SEM ACABAR PEGANDO UM TÍTULO VALIDO SEM QUERER
# POR ISSO DECIDI TIRAR SO OQ EU TINHA CERTEZA Q ERA SUJEIRA

# TENTEI TIRAR OPERAÇÕES MATEMÁTICAS, CARACTERES ESPECIAIS ESPECÍFICOS, DINHEIRO, NUMERO SOZINHO, CARACTERE SOZINHO, APOSTOFRO, ASPAS, DOIS PONTOS, SINAIS ARITMÉTRICOS
#   MAS ISSO ME LEVOU A REMOVER ALÉM DOS TÍTULOS QUE EU QUERIA, TAMBÉM ALGUNS TÍTULOS VALIDOS
#   DENTRO DESSE PANORAMA TAMBÉM HAVIA TÍTULOS QUE EU NÃO QUERIA, MAS QUE NÃO ERAM SUJEIRA, POR ISSO DECIDI TIRAR A SUJEIRA QUE EU TINHA CERTEZA QUE ERA SUJEIRA
# DESSA FORMA, TAMBEM APLIQUEI AS MESMAS CORREÇÕES TITULO ORIGINAL, FRASE DIVULGAÇÃO E SINOPSE

# "\" solto no inicio = residuo de escape de aspas do CSV original, so removemos o caractere
# Aspas duplicadas ou barra no meio/fim = geralmente linha inteira do CSV vazou pro campo titulo, nulificamos

# ATENÇÃO: esta célula NÃO grava mais a tabela. As 3 células de correção (titulo, titulo_original/
# frase_divulgacao e sinopse) agora são ENCADEADAS: cada uma parte do resultado da anterior e só a
# última grava. Antes, cada uma partia de df_silver_info_filmes e regravava a tabela inteira, então
# uma desfazia a correção da outra -- só a última (sinopse) sobrevivia na tabela final.

# Removendo o "\" solto do início do título (sem tornar NULL, só limpa o caractere)
df_info_barra_limpa = df_silver_info_filmes.withColumn(
    "titulo",
    F.when(
        F.col("titulo").startswith("\\"),
        F.trim(F.expr("substring(titulo, 2)"))
    ).otherwise(F.col("titulo"))
)

# Verificando se ainda sobrou sujeira (aspas duplicadas ou barra no meio/fim)
titulo_suspeito = F.col("titulo").rlike(r'\\|""')

df_info_titulo_tratado = df_info_barra_limpa.withColumn(
    "titulo", F.when(titulo_suspeito, None).otherwise(F.col("titulo"))
)

qtd_removida_barra = df_silver_info_filmes.filter(F.col("titulo").startswith("\\")).count()
qtd_nulificada = df_info_barra_limpa.filter(titulo_suspeito).count()

print(f"Total de títulos com barra inicial removida (mantendo o resto do título): {qtd_removida_barra}")
print(f"Total de títulos ainda com sujeira estrutural grave, convertidos para NULL: {qtd_nulificada}")

display(df_info_titulo_tratado.limit(10))

Total de títulos com barra inicial removida (mantendo o resto do título): 45
Total de títulos ainda com sujeira estrutural grave, convertidos para NULL: 54


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
564096,null,null,null,null,null,null,Não Informado,null,null
1144991,#69 Samskar Colony,#69 Samskar Colony,2022-03-18,2022,116,te,Lançado,"Koushik, a teenage boy, moves to the city with his family as a tenant in the Samskar colony where he falls in love with the landlady Vaishali, a married woman.",null
584871,#AbroHilo,#AbroHilo,2019-02-23,2019,54,es,Lançado,"Humor shapes the way Spaniards interact on Twitter: all sorts of topics can be used to make a joke and many anonymous commentators can become celebrities and compete with professional comedians. But sometimes certain jokes that defy political correctness have a high price for those who dare to make them, jokes that can freeze the smiles of thousands of people whose prejudices can put an end to some very successful artistic careers.",Humor on Twitter told by its protagonists
614696,#Alive,#살아있다,2020-06-24,2020,98,ko,Lançado,"As a grisly virus rampages a city, a lone man stays locked inside his apartment, digitally cut off from seeking help and desperate to find a way out.",You must survive.
476424,#Captured,#Captured,2017-09-05,2017,81,en,Lançado,A zealous vigilante looking to clean the Internet of sin targets a group of teens that broadcast misadventures on their chatroom.,Cleansing the internet of all sin
1036134,#Clout,#Clout,2022-10-11,2022,140,en,Lançado,Four stories unfold in Atlanta around the city's top influencers and their dependence on social media and their thirst for online prominence.,What would you do for it?
751146,#FollowMe,#FollowMe,2020-10-07,2020,22,en,Lançado,A documentary that enters into the world of Iraqi social media influencers and follows their perilous journey as they fight for their rights.,null
503352,#FriendButMarried,#TemanTapiMenikah,2018-03-28,2018,102,id,Lançado,"Pining for his high school crush for years, a young man puts up his best efforts to move out of the friend zone until she reveals she's getting married.",null
655293,#FriendButMarried 2,#TemanTapiMenikah 2,2020-02-27,2020,104,id,Lançado,"As Ayu and Ditto finally transition from best friends to newlyweds, a quick pregnancy creates uncertainty for the future of their young marriage.",null
716733,#HandballStrive,#ハンド全力,2020-07-24,2020,108,ja,Lançado,"The school’s handball club is about to close down. Can social media bring it back to life? Masao Kiyota is a high school student living in Japan’s southern Kumamoto Prefecture. Lacking passion for anything in life, he spends his days like so many youth on his smartphone along with his childhood friend, Okamoto. One day, they upload a photo taken three years previously when they were both part of their school’s handball team. To their surprise, the post goes a little viral. Encouraged, they add the hashtag: “#Handball Full Power” and are swarmed with “likes” from around the country. Amidst the sudden social media attention, Masao and Okamoto set to resuscitating a nearly defunct men’s handball team.",null


In [0]:
# CORRIGINDO SUJEIRA RESIDUAL NO TITULO ORIGINAL E FRASE DIVULGAÇÃO

# Aplica a mesma correção (remove "\" residual do início, preservando o resto do texto)
# nas colunas titulo_original e frase_divulgacao
# Parte de df_info_titulo_tratado (resultado da célula anterior) pra não desfazer a correção do titulo
df_info_colunas_corrigidas = (
    df_info_titulo_tratado
    .withColumn("titulo_original",
        F.when(
            F.col("titulo_original").startswith("\\"),
            F.trim(F.expr("substring(titulo_original, 2)"))
        ).otherwise(F.col("titulo_original"))
    )
    .withColumn("frase_divulgacao",
        F.when(
            F.col("frase_divulgacao").startswith("\\"),
            F.trim(F.expr("substring(frase_divulgacao, 2)"))
        ).otherwise(F.col("frase_divulgacao"))
    )
)

qtd_titulo_original_corrigido = df_silver_info_filmes.filter(F.col("titulo_original").startswith("\\")).count()
qtd_frase_corrigida = df_silver_info_filmes.filter(F.col("frase_divulgacao").startswith("\\")).count()

print(f"Total de titulo_original corrigidos: {qtd_titulo_original_corrigido}")
print(f"Total de frase_divulgacao corrigidos: {qtd_frase_corrigida}")

# NOTA SOBRE TÍTULOS NULOS (decisão documentada):
# Cheguei a implementar aqui um fallback que, quando o titulo virava NULL por sujeira estrutural,
# usava o titulo_original no lugar. Recuperava 10 dos 54 casos.
# Ao VALIDAR o resultado, porém, vi que nesses registros o titulo_original também está
# contaminado por column shift -- o que entrava como "título" eram frases de sinopse
# (ex: "photos and archival footage", "involving quotes from the French composer Erik Satie").
# Ou seja, o fallback trocava um NULL honesto por um dado ativamente errado, que se propagaria
# para a dim_movies e para o gold_genai_movies_context.
# Decisão: manter os 54 títulos como NULL. O documento da atividade não exige tratamento de
# título e não proíbe nulos nessa coluna, e o gold_genai_movies_context já trata esse caso
# com o fallback textual "Título desconhecido".

display(df_info_colunas_corrigidas.select("titulo", "titulo_original", "frase_divulgacao").limit(10))

Total de titulo_original corrigidos: 2
Total de frase_divulgacao corrigidos: 147


titulo,titulo_original,frase_divulgacao
null,null,null
#69 Samskar Colony,#69 Samskar Colony,null
#AbroHilo,#AbroHilo,Humor on Twitter told by its protagonists
#Alive,#살아있다,You must survive.
#BKKY,#BKKY,null
#Captured,#Captured,Cleansing the internet of all sin
#Clout,#Clout,What would you do for it?
#FollowMe,#FollowMe,null
#FriendButMarried,#TemanTapiMenikah,null
#FriendButMarried 2,#TemanTapiMenikah 2,null


In [0]:
# CORRIGINDO A SINOPSE APLICANDO A MESMA CORREÇÃO DOS ANTERIORES

# Parte de df_info_colunas_corrigidas (resultado da célula anterior), fechando a cadeia:
# titulo -> titulo_original/frase_divulgacao -> sinopse. Só aqui a tabela é gravada, já com TODAS as correções.

# Remção de "\" residual do início da sinopse 
df_sinopse_sem_barra = df_info_colunas_corrigidas.withColumn(
    "sinopse",
    F.when(
        F.col("sinopse").startswith("\\"),
        F.trim(F.expr("substring(sinopse, 2)"))
    ).otherwise(F.col("sinopse"))
)

# Remoção do ">" (e espaços) do início da sinopse 
df_sinopse_corrigida = df_sinopse_sem_barra.withColumn(
    "sinopse",
    F.trim(F.regexp_replace(F.col("sinopse"), r"^>+\s*", ""))
)

qtd_barra_corrigida = df_silver_info_filmes.filter(F.col("sinopse").startswith("\\")).count()
qtd_gt_corrigida = df_sinopse_sem_barra.filter(F.col("sinopse").startswith(">")).count()

print(f"Total de sinopses com '\\' no início, corrigidas: {qtd_barra_corrigida}")
print(f"Total de sinopses com '>' no início, corrigidas: {qtd_gt_corrigida}")

# Regrava a tabela com TODAS as correções da cadeia (titulo, titulo_original, frase_divulgacao e sinopse)
df_sinopse_corrigida.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

print("Tabela silver.tb_info_filmes REGRAVADA com titulo, titulo_original, frase_divulgacao e sinopse corrigidos.")

# Confirmação final
df_confirmacao = spark.table(f"{silver_schema}.tb_info_filmes")
qtd_restante_barra = df_confirmacao.filter(F.col("sinopse").startswith("\\")).count()
qtd_restante_gt = df_confirmacao.filter(F.col("sinopse").startswith(">")).count()
qtd_restante_titulo_barra = df_confirmacao.filter(F.col("titulo").startswith("\\")).count()
print(f"sinopse ainda com '\\' no início (deve ser 0): {qtd_restante_barra}")
print(f"sinopse ainda com '>' no início (deve ser 0): {qtd_restante_gt}")
print(f"titulo ainda com '\\' no início (deve ser 0): {qtd_restante_titulo_barra}")

display(df_confirmacao.select("id_filme", "titulo", "sinopse").limit(100))

Total de sinopses com '\' no início, corrigidas: 4778
Total de sinopses com '>' no início, corrigidas: 0
Tabela silver.tb_info_filmes REGRAVADA com titulo, titulo_original, frase_divulgacao e sinopse corrigidos.
sinopse ainda com '\' no início (deve ser 0): 0
sinopse ainda com '>' no início (deve ser 0): 0
titulo ainda com '\' no início (deve ser 0): 0


id_filme,titulo,sinopse
564096,null,null
1098846,null,null
635654,null,null
998516,null,null
881535,null,null
466287,null,now a disabled felon
507648,null,null
1103247,null,null
406776,null,null
860292,#1915House,"A century of secrets are hidden behind the fresh paint and modern additions. Peeling back the layers is letting something escape, and he might not see it till it's too late."


In [0]:
df_conf = spark.table(f"{silver_schema}.tb_info_filmes")

total_titulo_nulo = df_conf.filter(F.col("titulo").isNull()).count()
recuperaveis = df_conf.filter(
    F.col("titulo").isNull() &
    F.col("titulo_original").isNotNull() &
    ~F.col("titulo_original").rlike(r'\\|""')
).count()

print(f"Filmes com titulo NULL: {total_titulo_nulo}")
print(f"Desses, com titulo_original íntegro (recuperáveis): {recuperaveis}")

Filmes com titulo NULL: 54
Desses, com titulo_original íntegro (recuperáveis): 10


### A tabela tb_info_filmes é gravada 4 vezes, porém as últimas 3 foram no intuito de fazer correções em dados ruídosos e sujos.
###     preferi deixar desas maneira invés de dar uma gravação apenas no fim por acreditar que assim seja mais simples de debugar algo futuramente se for necessário

### TABELA `silver.tb_cotacao_dolar`

Escolhida pra ser tratada primeiro que a financeiro_filmes pq a do financeiro depende da cotação

In [0]:
# CARREGANDO DADOS DA COTAÇÃO BRONZE 

df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")
display(df_bronze_cotacao)

cotacaoCompra,cotacaoVenda,dataHoraCotacao,ingestion_datetime
5.169,5.1696,2026-09-14 13:10:08.144425,2026-09-18T21:52:34.103Z
5.1484,5.149,2026-09-15 13:09:19.199664,2026-09-18T21:52:34.103Z
5.152,5.1527,2026-09-16 13:05:30.35873,2026-09-18T21:52:34.103Z
5.1515,5.1521,2026-09-17 13:03:21.858212,2026-09-18T21:52:34.103Z
5.1569,5.1575,2026-09-18 13:03:34.742036,2026-09-18T21:52:34.103Z
5.169,5.1696,2026-09-14 13:10:08.144425,2026-09-19T03:57:28.142Z
5.1484,5.149,2026-09-15 13:09:19.199664,2026-09-19T03:57:28.142Z
5.152,5.1527,2026-09-16 13:05:30.35873,2026-09-19T03:57:28.142Z
5.1515,5.1521,2026-09-17 13:03:21.858212,2026-09-19T03:57:28.142Z
5.1569,5.1575,2026-09-18 13:03:34.742036,2026-09-19T03:57:28.142Z


In [0]:
# EXECUTANDO O FORWARD FILL PEDIDO NO DOCUMENTO .pdf PRA RESOLVER OS DIAS SEM COTAÇÃO

# F.to_date(F.col("dataHoraCotacao")) - Pega só a data (ano-mês-dia) de uma coluna com data+hora junto (ex: "2026-09-14 13:10:08.144425" vira 2026-09-14)
df_cotacao_com_data = df_bronze_cotacao.withColumn(
    "data_cotacao", F.to_date(F.col("dataHoraCotacao"))    # "to_date" interpreta perfeitamente pq ja vem com formato padronizado da API
)

# Mesma lógica da deduplicação usada em tb_info_filmes, mantendo a cotação mais recente por data
# Deduplicação mantendo a cotação mais recente por data
df_cotacao_deduplicada = deduplicar_por_chave(df_cotacao_com_data, ["data_cotacao"])

display(df_cotacao_deduplicada.orderBy("data_cotacao"))

Antes da deduplicação: 20 linhas
Depois da deduplicação: 5 linhas


cotacaoCompra,cotacaoVenda,dataHoraCotacao,ingestion_datetime,data_cotacao
5.169,5.1696,2026-09-14 13:10:08.144425,2026-09-21T02:27:48.568Z,2026-09-14
5.1484,5.149,2026-09-15 13:09:19.199664,2026-09-21T02:27:48.568Z,2026-09-15
5.152,5.1527,2026-09-16 13:05:30.35873,2026-09-21T02:27:48.568Z,2026-09-16
5.1515,5.1521,2026-09-17 13:03:21.858212,2026-09-21T02:27:48.568Z,2026-09-17
5.1569,5.1575,2026-09-18 13:03:34.742036,2026-09-21T02:27:48.568Z,2026-09-18


In [0]:
# EXECUTANDO O FORWARD FILL PEDIDO NO DOCUMENTO .pdf PRA RESOLVER OS DIAS SEM COTAÇÃO

from datetime import datetime, timedelta

# Define o intervalo de dias com base em hoje, não nos dados que já vieram da API
# Isso garante que as extremidades (ex: um sábado "hoje") também entrem na sequência
# O dia sem cotação que entrar na sequencia será preenchido pelo Forward Fill depois

data_max_desejada = datetime.today().date()     # Coleta a data de hoje (dia da execução dnv kkkk)
data_min_desejada = data_max_desejada - timedelta(days=13)  # Uma margem de segurança pra cobrir possíveis finais de semana nas extremidades do intervalo

# Print de validação do intervalo
print(f"Intervalo desejado: {data_min_desejada} até {data_max_desejada}")


# Esse SQL gera uma lista de todas as datas entre duas datas (sem pular dias)
#   explode() - Serve pra dividir em diferentes linhas e nao ficar so um array gigante com as datas
df_todas_as_datas = spark.sql(f"""
    SELECT explode(sequence(to_date('{data_min_desejada}'), to_date('{data_max_desejada}'), interval 1 day)) AS data_cotacao
""")

# Print de validação tambem
print(f"Total de dias na sequência: {df_todas_as_datas.count()}")
display(df_todas_as_datas.orderBy("data_cotacao"))

Intervalo desejado: 2026-09-08 até 2026-09-21
Total de dias na sequência: 14


data_cotacao
2026-09-08
2026-09-09
2026-09-10
2026-09-11
2026-09-12
2026-09-13
2026-09-14
2026-09-15
2026-09-16
2026-09-17


In [0]:
# EXECUTANDO O FORWARD FILL PEDIDO NO DOCUMENTO .pdf PRA RESOLVER OS DIAS SEM COTAÇÃO

# how="left" - Mantém TODAS as linhas da esquerda (df_todas_as_datas, nossa sequência completa)
#   Pra cada data da esquerda, tenta achar uma linha correspondente na direita (mesma data_cotacao)
#   Se achar, cola as colunas da direita (cotacaoCompra, cotacaoVenda, ...) do lado
#   Se NAO achar (ex: fim de semana), mantém a linha mesmo assim, só que com as colunas da direita como NULL
#   Isso é diferente do "inner" join, que IGNORARIA (removeria) as linhas sem correspondência
#   Os NULLs precisam aparecer pq vão ser preenchidos com Forward Fill

df_cotacao_completa = df_todas_as_datas.join(
    df_cotacao_deduplicada,   # tabela da direita (dados reais e deduplicados, só dias úteis)
    on="data_cotacao",     # coluna em comum usada pra "casar" as linhas das duas tabelas
    how="left"             # tipo do join, mantendo tudo da esquerda
)

print(f"Total de dias no período: {df_todas_as_datas.count()}")
display(df_cotacao_completa.orderBy("data_cotacao"))

Total de dias no período: 14


data_cotacao,cotacaoCompra,cotacaoVenda,dataHoraCotacao,ingestion_datetime
2026-09-08,null,null,null,null
2026-09-09,null,null,null,null
2026-09-10,null,null,null,null
2026-09-11,null,null,null,null
2026-09-12,null,null,null,null
2026-09-13,null,null,null,null
2026-09-14,5.169,5.1696,2026-09-14 13:10:08.144425,2026-09-21T02:27:48.568Z
2026-09-15,5.1484,5.149,2026-09-15 13:09:19.199664,2026-09-21T02:27:48.568Z
2026-09-16,5.152,5.1527,2026-09-16 13:05:30.35873,2026-09-21T02:27:48.568Z
2026-09-17,5.1515,5.1521,2026-09-17 13:03:21.858212,2026-09-21T02:27:48.568Z


In [0]:
# EXECUTANDO O FORWARD FILL PEDIDO NO DOCUMENTO .pdf PRA RESOLVER OS DIAS SEM COTAÇÃO

# orderBy("data_cotacao") - mais antigo pro mais novo
# .rowsBetween(Window.unboundedPreceding, 0) - Entre "início de tudo" e 0 significa "até a linha atual

# Window ordenada por data, sem particionar (queremos "olhar para trás" na série toda)
janela_forward_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)

df_cotacao_preenchida = (
    df_cotacao_completa

    # Pega o ultimo valor nao NULL
    .withColumn("cotacaoCompra", F.last("cotacaoCompra", ignorenulls=True).over(janela_forward_fill))
    .withColumn("cotacaoVenda", F.last("cotacaoVenda", ignorenulls=True).over(janela_forward_fill))
)

# Display de confirmação
display(df_cotacao_preenchida.orderBy("data_cotacao"))

#                   ===========================================================================================================
# dataHoraCotacao E ingestion_datetime ESTÃOS SENDO MANTIDAS NULL NOS DIAS DE FINAL DE SEMANA/FERIADO PQ NAO TEM DADOS DA API, CONSEQUENTEMENTE NAO TEM DATA
#                   ===========================================================================================================

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacaoCompra,cotacaoVenda,dataHoraCotacao,ingestion_datetime
2026-09-08,null,null,null,null
2026-09-09,null,null,null,null
2026-09-10,null,null,null,null
2026-09-11,null,null,null,null
2026-09-12,null,null,null,null
2026-09-13,null,null,null,null
2026-09-14,5.169,5.1696,2026-09-14 13:10:08.144425,2026-09-21T02:27:48.568Z
2026-09-15,5.1484,5.149,2026-09-15 13:09:19.199664,2026-09-21T02:27:48.568Z
2026-09-16,5.152,5.1527,2026-09-16 13:05:30.35873,2026-09-21T02:27:48.568Z
2026-09-17,5.1515,5.1521,2026-09-17 13:03:21.858212,2026-09-21T02:27:48.568Z


In [0]:
# TRADUZINDO NOME DAS COLUNAS DO cotacao_dolar

# POR MOTIVOS DE ACHAR ÚTIL AS INFORMAÇÕES E POR QUESTÃO DE RASTREABILIDADE, OPTEI POR MANTER TODAS AS COLUNAS NA COLUNA COTACAO SILVER

df_silver_cotacao_dolar = (
    df_cotacao_preenchida
    .withColumnRenamed("cotacaoCompra", "valor_compra")
    .withColumnRenamed("cotacaoVenda", "valor_venda")
    .withColumnRenamed("dataHoraCotacao", "data_hora_cotacao_original")
    .withColumnRenamed("ingestion_datetime", "data_hora_ingestao")

    # Select de todas elas dessa vez, diferente da movie_info
    .select("data_cotacao", "valor_compra", "valor_venda", "data_hora_cotacao_original", "data_hora_ingestao")
)

# Checagens de qualidade (condição e duplicatas)
dq_check_unique("silver.tb_cotacao_dolar", "data_cotacao única", df_silver_cotacao_dolar, ["data_cotacao"])
dq_check("silver.tb_cotacao_dolar", "valor_compra não nulo", df_silver_cotacao_dolar, F.col("valor_compra").isNotNull())

# Gravando da tabela Silver (dessa vez com modo overwrite pq nao é na bronze append only)
df_silver_cotacao_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

print("Tabela silver.tb_cotacao_dolar gravada com sucesso.")
display(df_silver_cotacao_dolar.orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] silver.tb_cotacao_dolar | data_cotacao única | 0 chaves duplicadas de 14 linhas
[FAIL] silver.tb_cotacao_dolar | valor_compra não nulo | 6/14 linhas falharam
Tabela silver.tb_cotacao_dolar gravada com sucesso.


data_cotacao,valor_compra,valor_venda,data_hora_cotacao_original,data_hora_ingestao
2026-09-08,null,null,null,null
2026-09-09,null,null,null,null
2026-09-10,null,null,null,null
2026-09-11,null,null,null,null
2026-09-12,null,null,null,null
2026-09-13,null,null,null,null
2026-09-14,5.169,5.1696,2026-09-14 13:10:08.144425,2026-09-21T02:27:48.568Z
2026-09-15,5.1484,5.149,2026-09-15 13:09:19.199664,2026-09-21T02:27:48.568Z
2026-09-16,5.152,5.1527,2026-09-16 13:05:30.35873,2026-09-21T02:27:48.568Z
2026-09-17,5.1515,5.1521,2026-09-17 13:03:21.858212,2026-09-21T02:27:48.568Z


### TABELA `silver.tb_financeiro_filmes`

In [0]:
# CARREGANDO FINANCIALS BRONZE

df_bronze_financials = spark.table(f"{bronze_schema}.tb_movies_financials")

# Pritando os primeiros 30 elementos de budget e revenue pra identificar os padrões de sujeira nos dados
print("Amostra de valores distintos de budget:")
df_bronze_financials.select("budget").distinct().show(30, truncate=False)
print("Amostra de valores distintos de revenue:")
df_bronze_financials.select("revenue").distinct().show(30, truncate=False)


# EXISTEM OS FORMATOS NO BUDGET "149000000", "149.0 M", "USD 149000000", "$ 149000000", "149.0 K"
# NO REVENUE EXISTE "-149000000" E  "149000000"

Amostra de valores distintos de budget:
+-------------+
|budget       |
+-------------+
|149000000    |
|84.0M        |
|90.0M        |
|60000000     |
|USD 30000000 |
|120000000    |
|42000000     |
|35000000     |
|15800000     |
|32000000     |
|3000000      |
|75.0M        |
|9.0M         |
|48000000     |
|$ 111000000  |
|USD 250000000|
|19000000     |
|6500000      |
|5.0M         |
|38.0M        |
|USD 108000000|
|7400000      |
|$ 24350000   |
|8.0M         |
|$ 18000000   |
|USD 5000000  |
|4531653      |
|$ 11795877   |
|USD 11900000 |
|USD 24393503 |
+-------------+
only showing top 30 rows
Amostra de valores distintos de revenue:
+----------+
|revenue   |
+----------+
|2052415039|
|566652812 |
|82468705  |
|634151679 |
|529323962 |
|760098996 |
|-130423032|
|605425157 |
|125479266 |
|78988148  |
|92600000  |
|118587880 |
|173469516 |
|259900000 |
|167323950 |
|429800000 |
|48453605  |
|120989656 |
|133423964 |
|-146745280|
|49447308  |
|9923127   |
|29942746  |
|56996304  |

In [0]:
# DEDUPLICAÇÃO DA TABELA FINANCEIRA 
df_financials_deduplicado = deduplicar_por_chave(df_bronze_financials, ["id"])

Antes da deduplicação: 530825 linhas
Depois da deduplicação: 99006 linhas


In [0]:
# LIMPANDO AS COLUNAS BUDGET E REVENUE


# Função para limpar valores monetários
# Remove textos de ausência de dado, símbolos de moeda, espaços, converte abreviação 'M' (milhões) e trata negativos/zeros como NULL.
def limpar_valor_monetario(coluna):

    # F.when(condição, valor) - equivale ao if
    # .otherwise(coluna) - equivale ao else
    # F.trim() - remove espaços em branco no inicio e fim
    # F.lower() - padrao pra deixar tudo minúsculo
    # .isin() - Verifica se o conteúdo é igual a algum dos parametros passados na função
    # F.regexp_replace(coluna, padrão, substituto) - Busca o texto passado por parametro e troca pelo substituto
    # .endswith() - Seleciona aqueles que terminarem com a string passada por parametro
    # .rlike(padrão) - Baseia-se no padrão pra aceitar somente numeros
    
    # Normalização dos valores de determinada coluna que nao possuem valor agregado e atribui "none"/NULL a eles
    # Se possuir valor numérico, mantém o valor
    col_tratada = F.when(
        F.trim(F.lower(coluna)).isin("unknown", "não informado", "nao informado", "n/a", ""),
        None
    ).otherwise(coluna)

    # Realmente precisa da notação (r"\$") pra buscar cifrão, sem isso o '$' siginifica fim da linha
    sem_usd = F.regexp_replace(col_tratada, "USD", "")  # Remove "USD"
    sem_cifrao = F.regexp_replace(sem_usd, r"\$", "")   # Remove "$"
    col_limpa = F.trim(sem_cifrao)  # Remove espaços

    # Deixa tudo em maiúsculo pra facilitar
    col_upper = F.upper(col_limpa)
    termina_com_m = col_upper.endswith("M")
    termina_com_k = col_upper.endswith("K")

    # Remove 'M' e 'K' e virgula
    numero_sem_sufixo = F.regexp_replace(col_upper, "[MK]$", "")    # 10.0 K vira 10.0
    numero_sem_virgula = F.regexp_replace(numero_sem_sufixo, ",", "")   # 1,000,000 vira 1000000

    # So armazena na variável o texto sem vírgulas que realmente parecer um número
    eh_numero_valido = numero_sem_virgula.rlike(r"^-?\d+(\.\d+)?$")
    valor_numerico = F.when(eh_numero_valido, numero_sem_virgula.cast("double")).otherwise(None)

    # Realiza as multiplicações equivalentes ao 'M' e 'K' se forem necessarias
    valor_final = (
        F.when(termina_com_m, valor_numerico * 1000000)
        .when(termina_com_k, valor_numerico * 1000)
        .otherwise(valor_numerico)
    )

    # Transforma zero e negativos em NULL
    return F.when(valor_final > 0, valor_final).otherwise(None)


df_financials_limpo = (
    df_financials_deduplicado
    .withColumn("orcamento_usd", limpar_valor_monetario(F.col("budget")))
    .withColumn("receita_usd", limpar_valor_monetario(F.col("revenue")))
)

display(df_financials_limpo.select("budget", "orcamento_usd", "revenue", "receita_usd").limit(100))

budget,orcamento_usd,revenue,receita_usd
90.0M,9.0E7,426505244,4.26505244E8
0,null,0,null
0,null,0,null
337200,337200.0,0,null
N/A,null,18850674,1.8850674E7
0,null,Não Informado,null
46000000,4.6E7,-23737523,null
0,null,0,null
7800000,7800000.0,0,null
0,null,Unknown,null


In [0]:
# CALCULANDO VALORES EM REAL
# PARA ISSO, DECIDI USAR A COTAÇÃO MAIS RECENTE DISPONIVEL NA TABELA SILVER DE COTAÇÃO
# EMBORA SEJA "INCOERENTE" PRECIFICAR UM FILME DE 2016 COM A COTAÇÃO 2026, MAS É A COTAÇÃO QUE TENHO EM MÃOS

# Pega a cotação mais recente disponivel
df_cotacao_atual = spark.table(f"{silver_schema}.tb_cotacao_dolar")


# Filtra pelos dias com cotação != NULL, ordena degressivamente e pega o primeiro (mais recente)
cotacao_mais_recente = (
    df_cotacao_atual
    .filter(F.col("valor_compra").isNotNull())
    .orderBy(F.desc("data_cotacao"))
    .select("valor_compra")
    .first()["valor_compra"]
)

# Printa a cotação usada
print(f"Cotação mais recente utilizada: R$ {cotacao_mais_recente}")

# Faz o cambio para real arredondando para 2 casas decimais 
df_financials_com_brl = (
    df_financials_limpo
    # Limito tambem dolar em 2 casas so por precaução de ocasionalmente poder aparecer um valor decimal
    .withColumn("orcamento_usd", F.round(F.col("orcamento_usd"), 2))
    .withColumn("receita_usd", F.round(F.col("receita_usd"), 2))
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(cotacao_mais_recente), 2))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * F.lit(cotacao_mais_recente), 2))
)

display(df_financials_com_brl.select("orcamento_usd", "orcamento_brl", "receita_usd", "receita_brl").limit(30))

Cotação mais recente utilizada: R$ 5.1569


orcamento_usd,orcamento_brl,receita_usd,receita_brl
9.0E7,4.64121E8,4.26505244E8,2.19944489278E9
null,null,null,null
null,null,null,null
337200.0,1738906.68,null,null
null,null,1.8850674E7,9.721104075E7
null,null,null,null
4.6E7,2.372174E8,null,null
null,null,null,null
7800000.0,4.022382E7,null,null
null,null,null,null


In [0]:
# DERIVANDO AS COLUNAS DE LUCRO E MARGEM DE LUCRO PERCENTUAL COM FOI PEDIDO NO DOCUMENTO

# Mas nao entendi isso de evitar divisão por zero, no proprio documento falou " garanta que valores zerados ou negativos sejam tratados como ausentes."
# Isso nao significa que cada zero ja seria tratado como NULL? então seria pra evitar a divisão por NULL, certo?


df_financials_com_lucro = (
    df_financials_com_brl
    # Lucro é igual Receita - Orçamento 
    .withColumn("lucro_usd", F.round(F.col("receita_usd") - F.col("orcamento_usd"), 2))
    .withColumn("lucro_brl", F.round(F.col("receita_brl") - F.col("orcamento_brl"), 2))
    # Margem de Lucro percentual é (Lucro / Orçamento) * 100, evitando divisão por zero
    .withColumn("margem_lucro_percentual",
        F.when(F.col("orcamento_usd") > 0, F.round((F.col("lucro_usd") / F.col("orcamento_usd")) * 100, 2))
        .otherwise(None)
    )
)

display(df_financials_com_lucro.select(
    "orcamento_usd", "receita_usd", "lucro_usd", "lucro_brl", "margem_lucro_percentual"
).limit(30))

orcamento_usd,receita_usd,lucro_usd,lucro_brl,margem_lucro_percentual
9.0E7,4.26505244E8,3.36505244E8,1.73532389278E9,373.89
null,null,null,null,null
null,null,null,null,null
337200.0,null,null,null,null
null,1.8850674E7,null,null,null
null,null,null,null,null
4.6E7,null,null,null,null
null,null,null,null,null
7800000.0,null,null,null,null
null,null,null,null,null


In [0]:
# GUARDANDO NA TABELA silver.tb_financeiro_filmes

# DECIDI MANTER AS COLUNAS DE LUCRO E MARGEM DE LUCRO POIS INTERPRETEI Q ELAS TERIAM QUE ESTAR PRESENTES, UMA VEZ DERIVADAS "Derive as colunas de Lucro (Dólar/Real) e Margem de Lucro Percentual, garantindo que operaçõesaritméticas com valores ausentes não invalidem o resultado e evitando divisões por zero."

df_silver_financeiro_filmes = (
    df_financials_com_lucro
    .withColumnRenamed("id", "id_filme")
    .select(
        "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
        "lucro_usd", "lucro_brl", "margem_lucro_percentual"
    )
)

# Checagens de qualidade
dq_check_unique("silver.tb_financeiro_filmes", "id_filme único", df_silver_financeiro_filmes, ["id_filme"])
dq_check("silver.tb_financeiro_filmes", "id_filme não nulo", df_silver_financeiro_filmes, F.col("id_filme").isNotNull())

df_silver_financeiro_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

# Print da tabela pra confirmação
print("Tabela silver.tb_financeiro_filmes gravada com sucesso.")
display(df_silver_financeiro_filmes.limit(30))

[PASS] silver.tb_financeiro_filmes | id_filme único | 0 chaves duplicadas de 99006 linhas
[PASS] silver.tb_financeiro_filmes | id_filme não nulo | 0/99006 linhas falharam
Tabela silver.tb_financeiro_filmes gravada com sucesso.


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
38700,9.0E7,4.26505244E8,4.64121E8,2.19944489278E9,3.36505244E8,1.73532389278E9,373.89
42018,null,null,null,null,null,null,null
42330,null,null,null,null,null,null,null
45033,337200.0,null,1738906.68,null,null,null,null
50022,null,1.8850674E7,null,9.721104075E7,null,null,null
66534,null,null,null,null,null,null,null
68730,4.6E7,null,2.372174E8,null,null,null,null
77027,null,null,null,null,null,null,null
91639,7800000.0,null,4.022382E7,null,null,null,null
109442,null,null,null,null,null,null,null


In [0]:
# Verificando a quantidade de NULLS em budget e revenue pq achei uma quantidade mt grande

total = df_financials_deduplicado.count()
zeros_budget = df_financials_deduplicado.filter(F.col("budget") == "0").count()
zeros_revenue = df_financials_deduplicado.filter(F.col("revenue") == "0").count()

print(f"Total de filmes: {total}")
print(f"Filmes com budget = '0': {zeros_budget} ({zeros_budget/total*100:.1f}%)")
print(f"Filmes com revenue = '0': {zeros_revenue} ({zeros_revenue/total*100:.1f}%)")

Total de filmes: 99006
Filmes com budget = '0': 84982 (85.8%)
Filmes com revenue = '0': 86790 (87.7%)


### TABELA `silver.tb_metricas_engajamento`

In [0]:
# CARREGANDO E DEDUPLICANDO A TABELA tb_movies_metrics

df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_metrics_deduplicado = deduplicar_por_chave(df_bronze_metrics, ["id"])

# Prints pra dar uma olhada nas colunas popularity, vote_average e vote_count (filtrado por tudo que nao pareça um numero)
print("Valores de popularity que NÃO são números decimais válidos (padrão ponto):")
df_metrics_deduplicado.select("popularity").distinct() \
    .filter(~F.col("popularity").rlike(r"^-?\d+(\.\d+)?$")) \
    .show(60, truncate=False)

print("Valores de vote_average que NÃO são números decimais válidos:")
df_metrics_deduplicado.select("vote_average").distinct() \
    .filter(~F.col("vote_average").rlike(r"^-?\d+(\.\d+)?$")) \
    .show(60, truncate=False)

print("Valores de vote_count que NÃO são números INTEIROS válidos:")
df_metrics_deduplicado.select("vote_count").distinct() \
    .filter(~F.col("vote_count").rlike(r"^\d+$")) \
    .show(60, truncate=False)

Antes da deduplicação: 536820 linhas
Depois da deduplicação: 99013 linhas
Valores de popularity que NÃO são números decimais válidos (padrão ponto):
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|popularity                                                                                                                                                                                                                |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2,203                                                                                                                                                                                      

In [0]:
print("Valores de popularity que NÃO são números decimais válidos (padrão ponto):")
df_metrics_deduplicado.select("popularity").distinct() \
    .filter(~F.col("popularity").rlike(r"^-?\d+(\.\d+)?$")) \
    .show(200, truncate=False)

print("Valores de vote_average que NÃO são números decimais válidos:")
df_metrics_deduplicado.select("vote_average").distinct() \
    .filter(~F.col("vote_average").rlike(r"^-?\d+(\.\d+)?$")) \
    .show(200, truncate=False)

print("Valores de vote_count que NÃO são números INTEIROS válidos:")
df_metrics_deduplicado.select("vote_count").distinct() \
    .filter(~F.col("vote_count").rlike(r"^\d+$")) \
    .show(200, truncate=False)

# APENAS PRA VERIFICAR QUAIS SAO OS FORMATOS DE RUÍDOS NOS DADOS A SEREM TRATADOS

Valores de popularity que NÃO são números decimais válidos (padrão ponto):
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|popularity                                                                                                                                                                                                                                                                                                                                                                                                                |
+--------------------------------------------------------------------------------------------------

In [0]:
# CRIANDO FUNÇÕES 

# Função pra Normalizar os valores decimais
def limpar_valor_decimal(coluna, minimo=None, maximo=None):
    # Toda vírgula vira ponto
    col_com_ponto = F.regexp_replace(coluna, ",", ".")      

    # Valida se é número decimal válido
    eh_numero_valido = col_com_ponto.rlike(r"^-?\d+(\.\d+)?$")      

    # Converte o numero pra double se passar na validação regex
    valor_numerico = F.when(eh_numero_valido, col_com_ponto.cast("double")).otherwise(None)     

    # Só mantém se for maior ou igual a zero, diferente disso vira NULL
    valor_numerico = F.when(valor_numerico >= 0, valor_numerico).otherwise(None)

    # Torna NULL qualquer numero fora do intervalo passado na função
    if (minimo is not None) and (maximo is not None):
        valor_numerico = F.when(
            (valor_numerico >= minimo) & (valor_numerico <= maximo), valor_numerico
        ).otherwise(None)

    return valor_numerico


# Função pra tornar NULL qualquer numero não inteiro
def limpar_valor_inteiro(coluna):
    # Criado pq contagem de voto n tem como ter decimal, n existe algo ter 2,3 votos, ou tem 2 ou tem 3
    eh_inteiro_valido = coluna.rlike(r"^\d+$")
    valor_numerico = F.when(eh_inteiro_valido, coluna.cast("int")).otherwise(None)
    return valor_numerico


# Guardando no dataframe com os valores limpos (usando as funções nas colunas que precisam delas)
df_metrics_limpo = (
    df_metrics_deduplicado
    .withColumn("popularidade", limpar_valor_decimal(F.col("popularity")))
    .withColumn("nota_media_tmdb", limpar_valor_decimal(F.col("vote_average"), minimo=0, maximo=10))
    .withColumn("qtd_votos_tmdb", limpar_valor_inteiro(F.col("vote_count")))
    .withColumn("nota_media_imdb", limpar_valor_decimal(F.col("averageRating"), minimo=0, maximo=10))
    .withColumn("qtd_votos_imdb", limpar_valor_inteiro(F.col("numVotes")))
)

# CORREÇÃO EXTRA (column shift): "popularidade" contaminada com valores de ANO.
# Detectado ao validar a Pergunta 2 da Gold (top 5 popularidade): filmes como
# "wwe survivor series 2018" e "La Fellinette" apareciam com popularidade EXATAMENTE
# igual ao ano de lançamento (2018 e 2020). Popularidade real do TMDB nunca é um número
# inteiro "redondo" (sempre vem com várias casas decimais, ex: 2994.357) — então um valor
# inteiro exato dentro de uma faixa plausível de ano (1880-2030) é sinal de contaminação
# de coluna, não popularidade de verdade. Tratamos como NULL.
eh_valor_suspeito_de_ano = (
    (F.col("popularidade") == F.floor(F.col("popularidade"))) &   # é inteiro exato (sem casas decimais)
    (F.col("popularidade").between(1880, 2030))                   # e cai numa faixa plausível de ano
)

df_metrics_limpo = df_metrics_limpo.withColumn(
    "popularidade",
    F.when(eh_valor_suspeito_de_ano, None).otherwise(F.col("popularidade"))
)

# Display de confirmação
display(df_metrics_limpo.select(
    "popularity", "popularidade", "vote_average", "nota_media_tmdb",
    "vote_count", "qtd_votos_tmdb"
).limit(60))

popularity,popularidade,vote_average,nota_media_tmdb,vote_count,qtd_votos_tmdb
3.403,3.403,6.63,6.63,27,27
1.561,1.561,10.0,10.0,1,1
1.837,1.837,52.0,null,9,9
28.9,28.9,7.119,7.119,2822,2822
3.508,3.508,5.7,5.7,11,11
2.018,2.018,8.5,8.5,2,2
6.813,6.813,3.0,3.0,9,9
24.75,24.75,6.205,6.205,3369,3369
62.006,62.006,6.648,6.648,11176,11176
2.019,2.019,5.0,5.0,5,5


In [0]:
display(df_metrics_limpo.select(
    "averageRating", "nota_media_imdb", "numVotes", "qtd_votos_imdb"
).limit(30))

averageRating,nota_media_imdb,numVotes,qtd_votos_imdb
6.8,6.8,1750,1750
6.4,6.4,34,34
5.8,5.8,208,208
7.2,7.2,135908,135908
5.3,5.3,562,562
null,null,177,177
2.2,2.2,384,384
null,null,120814,120814
6.5,6.5,391135,391135
5.3,5.3,574,574


In [0]:
# GRAVANDO NA SILVER A TABELA metricas_engajamento

df_silver_metricas_engajamento = (
    df_metrics_limpo
    .withColumnRenamed("id", "id_filme")
    .select(
        "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb"
    )
)


# Checagens de qualidade

# Checando se so tem uma linha por filme
dq_check_unique("silver.tb_metricas_engajamento", "id_filme único", df_silver_metricas_engajamento, ["id_filme"])

# Checando se tem nota null
dq_check("silver.tb_metricas_engajamento", "id_filme não nulo", df_silver_metricas_engajamento, F.col("id_filme").isNotNull())

# Checando tbm com base na regra de negócio (n pode ter nota fora do intervalo de 0 a 10)
dq_check("silver.tb_metricas_engajamento", "nota_media_tmdb dentro do intervalo 0-10",
         df_silver_metricas_engajamento,
         F.col("nota_media_tmdb").isNull() | ((F.col("nota_media_tmdb") >= 0) & (F.col("nota_media_tmdb") <= 10)))

# Mesmo esquema de gravar com overwrite
df_silver_metricas_engajamento.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

# Print de confirmação
print("Tabela silver.tb_metricas_engajamento gravada com sucesso.")
display(df_silver_metricas_engajamento.limit(20))

[PASS] silver.tb_metricas_engajamento | id_filme único | 0 chaves duplicadas de 99013 linhas
[PASS] silver.tb_metricas_engajamento | id_filme não nulo | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | nota_media_tmdb dentro do intervalo 0-10 | 0/99013 linhas falharam
Tabela silver.tb_metricas_engajamento gravada com sucesso.


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
42018,3.403,6.63,27,6.8,1750
42330,1.561,10.0,1,6.4,34
66534,1.837,null,9,5.8,208
68730,28.9,7.119,2822,7.2,135908
91639,3.508,5.7,11,5.3,562
124346,2.018,8.5,2,null,177
146834,6.813,3.0,9,2.2,384
153518,24.75,6.205,3369,null,120814
166426,62.006,6.648,11176,6.5,391135
167859,2.019,5.0,5,5.3,574


### TABELA `silver.tb_movies_reviews`

In [0]:
# CARREGANDO TABELA tb_movies_reviews

df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

print(f"Total de linhas na Bronze: {df_bronze_reviews.count()}")

# Printando as diferentes notas em ordem crescente
print("Amostra de valores distintos de nota:")
df_bronze_reviews.select("nota").distinct().orderBy("nota").show(200, truncate=False)

# Filtrando e dando select nos comentarios vazios pra ter certeza se existem
print("Comentários vazios/em branco:")
df_bronze_reviews.filter(
    F.col("comentario").isNull() | (F.trim(F.col("comentario")) == "")
).select("comentario").show(30, truncate=False)


# TODAS AS NOTAS ESTÃO NO INTERVALO CORRETO, MAS POR PRECAUÇÃO VOU MANTER A VALIDAÇÃO DO INTERVALO ENTRE 0 E 10
# SIM, EXISTEM COMENTARIOS VAZIOS

Total de linhas na Bronze: 162060
Amostra de valores distintos de nota:
+----+
|nota|
+----+
|NULL|
|0.0 |
|0.1 |
|0.2 |
|0.3 |
|0.4 |
|0.5 |
|0.6 |
|0.7 |
|0.8 |
|0.9 |
|1.0 |
|1.1 |
|1.2 |
|1.3 |
|1.4 |
|1.5 |
|1.6 |
|1.7 |
|1.8 |
|1.9 |
|2.0 |
|2.1 |
|2.2 |
|2.3 |
|2.4 |
|2.5 |
|2.6 |
|2.7 |
|2.8 |
|2.9 |
|3.0 |
|3.1 |
|3.2 |
|3.3 |
|3.4 |
|3.5 |
|3.6 |
|3.7 |
|3.8 |
|3.9 |
|4.0 |
|4.1 |
|4.2 |
|4.3 |
|4.4 |
|4.5 |
|4.6 |
|4.7 |
|4.8 |
|4.9 |
|5.0 |
|5.1 |
|5.2 |
|5.3 |
|5.4 |
|5.5 |
|5.6 |
|5.7 |
|5.8 |
|5.9 |
|6.0 |
|6.1 |
|6.2 |
|6.3 |
|6.4 |
|6.5 |
|6.6 |
|6.7 |
|6.8 |
|6.9 |
|7.0 |
|7.1 |
|7.2 |
|7.3 |
|7.4 |
|7.5 |
|7.6 |
|7.7 |
|7.8 |
|7.9 |
|8.0 |
|8.1 |
|8.2 |
|8.3 |
|8.4 |
|8.5 |
|8.6 |
|8.7 |
|8.8 |
|8.9 |
|9.0 |
|9.1 |
|9.2 |
|9.3 |
|9.4 |
|9.5 |
|9.6 |
|9.7 |
|9.8 |
|9.9 |
|10.0|
+----+

Comentários vazios/em branco:
+----------+
|comentario|
+----------+
|NULL      |
|NULL      |
|NULL      |
|NULL      |
|NULL      |
|NULL      |
|NULL      |
|NULL      |
|NULL      |

In [0]:
# DEDUPLICAÇÃO DE LINHAS TOTALMENTE IGUAIS, VALIDANDO NOTAS E NORMALIZANDO COMENTARIOS

# A DEDUPLICAÇÃO É FEITA DIFERENTE E CONSIDERANDO TODAS AS COLUNAS
#   PQ UM FILME PODE TER DIVERSAS AVALIAÇÕES, E ESSAS AVALIAÇÕES TEM QUE ESTAR VINCULADAS AO ID DO FILME, SE DEDUPLICASSE SOMENTE PELO ID MUITO SE PERDERIA
#   POR ISSO, CONSIDERAR TAMBEM NOME, NOTA E COMENTÁRIO, PQ O COMENTARIO PODE SER O MESMO
# A deduplicação considera todas as colunas de uma linha, nao só o ID e data
df_reviews_deduplicado = df_bronze_reviews.dropDuplicates(["id", "nome", "nota", "comentario"])

# Print de validação da deduplicação
print(f"Antes da dedup: {df_bronze_reviews.count()} linhas")
print(f"Depois da dedup: {df_reviews_deduplicado.count()} linhas")

df_reviews_limpo = (
    df_reviews_deduplicado
    # Mesmo com as notas vistas que estão no intervalo, manter a regra de negócio do intervalo de 0 a 10 para possiveis novos dados
    .withColumn("nota_validada",
        F.when((F.col("nota") >= 0) & (F.col("nota") <= 10), F.col("nota")).otherwise(None)
    )
    # Tornando os comentarios vazios, em branco ou NULL viram "Sem comenrário"
    .withColumn("comentario_tratado",
        F.when(
            F.col("comentario").isNull() | (F.trim(F.col("comentario")) == ""),
            "Sem comentário"
        ).otherwise(F.trim(F.col("comentario")))
    )
)

# Display pra validar
display(df_reviews_limpo.select("nota", "nota_validada", "comentario", "comentario_tratado").limit(30))

Antes da dedup: 162060 linhas
Depois da dedup: 32412 linhas


nota,nota_validada,comentario,comentario_tratado
4.4,4.4,null,Sem comentário
7.6,7.6,Legal! Uma boa opção para o fim de semana.,Legal! Uma boa opção para o fim de semana.
6.1,6.1,Assisti até o final mas não me marcou.,Assisti até o final mas não me marcou.
2.1,2.1,Horrível! Perda de tempo.,Horrível! Perda de tempo.
3.5,3.5,null,Sem comentário
3.2,3.2,"Fraco, não recomendo.","Fraco, não recomendo."
9.0,9.0,Excepcional! História envolvente e atuações impecáveis.,Excepcional! História envolvente e atuações impecáveis.
9.8,9.8,Perfeito! Um dos melhores filmes que já vi.,Perfeito! Um dos melhores filmes que já vi.
9.1,9.1,"Obra-prima do cinema, simplesmente espetacular.","Obra-prima do cinema, simplesmente espetacular."
7.7,7.7,Filme interessante com boas atuações.,Filme interessante com boas atuações.


In [0]:
# CRIANDO FUNÇÕES 

# Função pra Normalizar os valores decimais
def limpar_valor_decimal(coluna, minimo=None, maximo=None):
    # Toda vírgula vira ponto
    col_com_ponto = F.regexp_replace(coluna, ",", ".")      

    # Valida se é número decimal válido
    eh_numero_valido = col_com_ponto.rlike(r"^-?\d+(\.\d+)?$")      

    # Converte o numero pra double se passar na validação regex
    valor_numerico = F.when(eh_numero_valido, col_com_ponto.cast("double")).otherwise(None)     

    # Só mantém se for maior ou igual a zero, diferente disso vira NULL
    valor_numerico = F.when(valor_numerico >= 0, valor_numerico).otherwise(None)

    # Torna NULL qualquer numero fora do intervalo passado na função
    if (minimo is not None) and (maximo is not None):
        valor_numerico = F.when(
            (valor_numerico >= minimo) & (valor_numerico <= maximo), valor_numerico
        ).otherwise(None)

    return valor_numerico


# Função pra tornar NULL qualquer numero não inteiro
def limpar_valor_inteiro(coluna):
    # Criado pq contagem de voto n tem como ter decimal, n existe algo ter 2,3 votos, ou tem 2 ou tem 3
    eh_inteiro_valido = coluna.rlike(r"^\d+$")
    valor_numerico = F.when(eh_inteiro_valido, coluna.cast("int")).otherwise(None)
    return valor_numerico


# Guardando no dataframe com os valores limpos (usando as funções nas colunas que precisam delas)
df_metrics_limpo = (
    df_metrics_deduplicado
    .withColumn("popularidade", limpar_valor_decimal(F.col("popularity")))
    .withColumn("nota_media_tmdb", limpar_valor_decimal(F.col("vote_average"), minimo=0, maximo=10))
    .withColumn("qtd_votos_tmdb", limpar_valor_inteiro(F.col("vote_count")))
    .withColumn("nota_media_imdb", limpar_valor_decimal(F.col("averageRating"), minimo=0, maximo=10))
    .withColumn("qtd_votos_imdb", limpar_valor_inteiro(F.col("numVotes")))
)

# CORREÇÃO EXTRA (column shift): "popularidade" contaminada com valores de ANO!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# Detectado ao validar a Pergunta 2 da Gold (top 5 popularidade): filmes como "wwe survivor series 2018" e "La Fellinette" apareciam com popularidade EXATAMENTE
# igual ao ano de lançamento (2018 e 2020). Popularidade real do TMDB nunca é um número inteiro "redondo" (sempre vem com várias casas decimais, ex: 2994.357), então um valor
# inteiro exato dentro de uma faixa plausível de ano (1880-2030) é sinal de contaminação de coluna, não popularidade de verdade. Tratamos como NULL.
eh_valor_suspeito_de_ano = (
    (F.col("popularidade") == F.floor(F.col("popularidade"))) &   # é inteiro exato (sem casas decimais)
    (F.col("popularidade").between(1880, 2030))                   # e cai numa faixa plausível de ano
)

df_metrics_limpo = df_metrics_limpo.withColumn(
    "popularidade",
    F.when(eh_valor_suspeito_de_ano, None).otherwise(F.col("popularidade"))
)

# Display de confirmação
display(df_metrics_limpo.select(
    "popularity", "popularidade", "vote_average", "nota_media_tmdb",
    "vote_count", "qtd_votos_tmdb"
).limit(60))

popularity,popularidade,vote_average,nota_media_tmdb,vote_count,qtd_votos_tmdb
3.403,3.403,6.63,6.63,27,27
1.561,1.561,10.0,10.0,1,1
1.837,1.837,52.0,null,9,9
28.9,28.9,7.119,7.119,2822,2822
3.508,3.508,5.7,5.7,11,11
2.018,2.018,8.5,8.5,2,2
6.813,6.813,3.0,3.0,9,9
24.75,24.75,6.205,6.205,3369,3369
62.006,62.006,6.648,6.648,11176,11176
2.019,2.019,5.0,5.0,5,5


### TABELA `silver.tb_generos`

In [0]:
# CARREGA TABELA bronze.tb_credits_and_tags E DEDUPLICA OS DADOS

df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_credits_deduplicado = deduplicar_por_chave(df_bronze_credits, ["id"])

# Print pra ver as diferentes represesntações
print("Amostra de valores distintos de genres:")
df_credits_deduplicado.select("genres").distinct().show(200, truncate=False)

# OS SEPARADORES SÃO ',' '|' ';'
# ALÉM DA SUJEIRA textos corridos, numeros, arquivos .jpeg e algumas aspas

Antes da deduplicação: 531600 linhas
Depois da deduplicação: 99006 linhas
Amostra de valores distintos de genres:
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|genres                                                                                                                                                                                                          |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Thriller|Mystery|Drama|Crime                                                                                                                                                                                    |
|History, Documentary                     

In [0]:
# NORMALIZAR SEPARADOES DE GENEROS, "EXPLODIR" E FILTRAR GENEREOS VALIDOS

# OPTEI POR FAZER UMA LISTA COM OS GENEROS EXISTENTES BASEADO NO PROPRIO TMDB PRA DPS SO COMPARAR E PEGAR OS VÁLIDOS
GENEROS_VALIDOS = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "Thriller", "TV Movie", "War", "Western"
]

# Explodindo os generos
# Precisa ser feito de modo q uma linha ID filme com diferentes generos virem varias linhas com o mesmo ID filme, porém com genero diferente
df_generos_explodido = (
    df_credits_deduplicado

    # Normalizando os separadores '|' ';' para virarem vírgula ','
    #   "[|;]" serve como um "qualquer um desses dois caracteres"
    .withColumn("genres_normalizado", F.regexp_replace(F.col("genres"), r"[|;]", ","))  # Mantive vírgula como padrão pra facilitar

    # Dividindo o texto em uma lista (vírgula como separador)
    .withColumn("genero_lista", F.split(F.col("genres_normalizado"), ","))  # "Action, Comedy" vira ["Action", "Comedy"]

    # Transformando cada item da lista em uma linha própria
    #   Se um filme tinha 2 gêneros na mesma linha, essa linha viram 2, uma para cada gênero e todas com o mesmo id (pq é o msm filme)
    .select("id", F.explode(F.col("genero_lista")).alias("genero_bruto"))

    # Removendo espaços em branco e aspas que sobraram com o F.trim()
    .withColumn("genero_limpo", F.trim(F.regexp_replace(F.col("genero_bruto"), '"', "")))

    # Filtrando os gêneros inválidos (Aqueles que nao estão na lista GENEROS_VALIDOS)
    .filter(F.col("genero_limpo").isin(GENEROS_VALIDOS))

    # Removendo duplicatas 
    # Pra caso exista algum registro com "Drama, Drama, Drama" por exemplo
    .select("id", "genero_limpo").distinct()
)

# Print de validação de linhas e display
print(f"Total de linhas (filme x gênero): {df_generos_explodido.count()}")
display(df_generos_explodido.orderBy("id").limit(30))

Total de linhas (filme x gênero): 142149


id,genero_limpo
14564,Horror
32471,Family
32471,Comedy
38492,Music
38700,Thriller
38700,Action
38700,Crime
42018,Drama
42330,Fantasy
42330,Animation


In [0]:
# GRAVANDO TABELA silver.tb_generos

df_silver_generos = (
    df_generos_explodido
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("genero_limpo", "genero")
    .select("id_filme", "genero")
)

# Checagens de qualidade
# Checando se nao tem ID de filme NULL
dq_check("silver.tb_generos", "id_filme não nulo", df_silver_generos, F.col("id_filme").isNotNull())

# Checando se nao tem genero NULL
dq_check("silver.tb_generos", "genero não nulo", df_silver_generos, F.col("genero").isNotNull())

# Checando ID junto com o genero, ja q podem existir varias linhas com o mesmo ID de filme, mas nao o mesmo ID com o mesmo genero
dq_check_unique("silver.tb_generos", "combinação id_filme + genero única", df_silver_generos, ["id_filme", "genero"])

# Padrão de gravação com overwrite
df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

# Print e display de validação
print("Tabela silver.tb_generos gravada com sucesso.")
display(df_silver_generos.orderBy("id_filme").limit(50))

[PASS] silver.tb_generos | id_filme não nulo | 0/142149 linhas falharam
[PASS] silver.tb_generos | genero não nulo | 0/142149 linhas falharam
[PASS] silver.tb_generos | combinação id_filme + genero única | 0 chaves duplicadas de 142149 linhas
Tabela silver.tb_generos gravada com sucesso.


id_filme,genero
14564,Horror
32471,Family
32471,Comedy
38492,Music
38700,Action
38700,Crime
38700,Thriller
42018,Drama
42330,Fantasy
42330,Adventure


### TABELA `silver.tb_pessoas_empresas`

In [0]:
# CHECANDO QUAIS AS POSSIBILIDADES PRA CADA CAST, DIRECTORS, WRITERS E PRODUCTION COMPANIES

# Explodindo os cast
# Precisa ser feito de modo q uma linha ID filme com diferentes pessoas da empresa virem varias linhas com o mesmo ID filme, porém com pessoas diferentes

print("Amostra de valores distintos de cast:")
df_credits_deduplicado.select("cast").distinct().show(100, truncate=False)

print("Amostra de valores distintos de directors:")
df_credits_deduplicado.select("directors").distinct().show(100, truncate=False)

print("Amostra de valores distintos de writers:")
df_credits_deduplicado.select("writers").distinct().show(100, truncate=False)

print("Amostra de valores distintos de production_companies:")
df_credits_deduplicado.select("production_companies").distinct().show(100, truncate=False)

# OBTIVE RUÍDOS COMO NUMEROS, ARQUIVOS .jpeg, GENEROS, NOMES DE PAÍSES(??) e TEXTOS CORRIDOS

Amostra de valores distintos de cast:
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|cast                                                                                                                                                                                                                 |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Kevin Heffernan, Steve Lemme, Erik Stolhanske, Emmanuelle Chriqui, Lynda Carter, Rob Lowe, Marisa Coughlan, Brian Cox, Tyler Labine, Will Sasso                                                                      |
|Song Yang, Yu Chenghui, Chengyuan Li, Zhao Zheng, Wang Yanni, Li Chang-Lin, Ma Jun               

In [0]:
# NORMALIZANDO E EXPLODINDO AS 4 COLUNAS

# Decidi criar uma função ja q vou precisar aplicar o mesmo processo 4x, uma pra cada coluna
def normalizar_e_explodir(df, coluna_origem, tipo_entidade):
    return (
        df
        # Substituindo '|' e ';' por ','
        .withColumn("col_normalizada", F.regexp_replace(F.col(coluna_origem), r"[|;]", ","))
        
        # Criando uma lista separada por vírgula igual com os generos
        .withColumn("lista", F.split(F.col("col_normalizada"), ","))
        
        # Explodindo a lista
        .select("id", F.explode(F.col("lista")).alias("entidade_bruta"))
        
        # Removendo aspas e espaços desnecessários com F.trim()
        .withColumn("entidade_limpa", F.trim(F.regexp_replace(F.col("entidade_bruta"), '"', "")))
        
        # Marcando o tipo de entidade
        .withColumn("tipo_entidade", F.lit(tipo_entidade))
        
        # Filtrando pra remover NULL ja aqui (se o tipo for diferente de NULL)
        .filter(F.col("entidade_limpa") != "")
    )

# Aplicando a função
df_cast_exp = normalizar_e_explodir(df_credits_deduplicado, "cast", "Ator") 
df_directors_exp = normalizar_e_explodir(df_credits_deduplicado, "directors", "Diretor")
df_writers_exp = normalizar_e_explodir(df_credits_deduplicado, "writers", "Roteirista")
df_companies_exp = normalizar_e_explodir(df_credits_deduplicado, "production_companies", "Produtora")

# .union() - Permite empilhar dois dataframes do mesmo schema

# Juntando cast + directors num DataFrame só
df_cast_e_directors = df_cast_exp.union(df_directors_exp)

# Juntando (cast + directors) + writers num DataFrame só
df_com_writers = df_cast_e_directors.union(df_writers_exp)

# Juntando (cast + directors + writers) + companies num DataFrame só
df_todas_entidades = df_com_writers.union(df_companies_exp)

# Print pra verificar a quantidade de linhas pós .union()
print(f"Total de linhas (todas entidades, ainda com sujeira): {df_todas_entidades.count()}")

# Isolando números puros, .jpeg, textos corridos
print("Amostra de valores suspeitos (números, paths, textos longos):")
df_todas_entidades.select("entidade_limpa").distinct().filter(
    F.col("entidade_limpa").rlike(r"^-?\d+([.,]\d+)?,?$") |    # Números
    F.col("entidade_limpa").startswith("/") |                  # .jpeg
    (F.length(F.col("entidade_limpa")) > 60)                   # Texto corrido
).show(200, truncate=False)

Total de linhas (todas entidades, ainda com sujeira): 910763
Amostra de valores suspeitos (números, paths, textos longos):
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|entidade_limpa                                                                                                                                                                                                                                                                  |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|7.8                                                

In [0]:
# LIMPANDO A SUJEIRA ENCONTRADA (numero, .jpeg e texto corrido), NORMALIZANDO E GRAVANDO silver.tb_pessoas_empresas

# Criando placeholders pra serem filtrados da tabela
PALAVRAS_PLACEHOLDER = ["nenhum", "none", "unknown", "n/a", "na", "null", "vazio", "sem nome"]

# Filtro pra remover sujeira uso de ~() pra pegar o contrario do .filter()
# Decidi filtrar a entidade por maior que 2 e menor que 60 caracteres pra evitar que frases/sinopses (que costumam ultrapassar 60) entrem como ruído ou sujeira na minha coluna
df_entidades_limpo = df_todas_entidades.filter(
    ~(
        F.col("entidade_limpa").rlike(r"^-?\d+([.,]\d+)?,?$") |    # Números
        F.col("entidade_limpa").startswith("/") |                  # .jpeg
        (F.length(F.col("entidade_limpa")) > 60) |                 # Texto corrido
        (F.length(F.col("entidade_limpa")) <= 2) |                 # Valores mt curtos
        ~F.col("entidade_limpa").rlike(r"\p{L}") |                 # NENHUMA letra de nenhum alfabeto
        F.lower(F.col("entidade_limpa")).isin(PALAVRAS_PLACEHOLDER) | # Palavras placeholder (achei um "nenhum" olhando a coluna)

        # Encontrei padrões de década (ex: "1990s") e século (ex: "12th Century") na coluna de pessoas -
        # não são nomes, são vazamento de texto de sinopse/tagline pra coluna errada (column shift)
        # Regex ANCORADA (^...$), ou seja, só bate se a string inteira for EXATAMENTE esse padrão, sem mais nada.
        # Isso é proposital: evita pegar nomes artísticos reais com número (ex: "50 Cent", "21 Savage", "9m88")
        # e produtoras reais tipo "20th Century Studios"/"20th Century Fox" (que têm palavra extra depois de "Century")
        F.col("entidade_limpa").rlike(r"(?i)^\d{4}s$|^\d{1,2}(st|nd|rd|th) century$")
    )
)

# Padronização de capitalização (como foi pedido no documento) e remoção de duplicatas
#   .initcap() - Padroniza a capitalização para "Title Case"
df_silver_pessoas_empresas = (
    df_entidades_limpo
    
    # Achei alguns erros de linhas com '+' ou '/' no meio do nome, troquei por espaço pq o initcap nao capitaliza esses simbolos
    .withColumn("entidade_para_capitalizar", F.regexp_replace(F.col("entidade_limpa"), r"[/+]", " "))

    # Encontrei nomes reais começando com aspa simples, tipo 'weird Al' Yankovic (confirmei pesquisando no google)
    # e 'wáats'asdíyei Joe Yates (tbm confirmado via google)
    # A aspa em si não é ruído, por isso deve ficar (caso seja tirada, alguns nomes reais podem ficar errado)
    # O problema é que o .initcap() só reconhece ESPAÇO como separador de palavra 
    # Então numa string tipo "'weird Al", a aspa fica grudada no "w" e o initcap capitaliza só o primeiro caractere da "palavra" ('), deixando o "weird" errado (minúsculo)
    # Por isso, pra nomes que começam com aspa, o nome é capitalizado sem a aspas e dps ela é adicionada novamente
    .withColumn("entidade_capitalizada",
        F.when(
            F.col("entidade_para_capitalizar").startswith("'"),
            F.concat(F.lit("'"), F.initcap(F.expr("substring(entidade_para_capitalizar, 2)")))
        ).otherwise(F.initcap(F.col("entidade_para_capitalizar")))
    )

    # Devolve os espaços trocados de volta para '/' ou '+' originais
    .withColumn("nome_padronizado",
        F.regexp_replace(
            F.regexp_replace(F.col("entidade_capitalizada"), r"(?<=\w) / (?=\w)", "/"),
            r"(?<=\w) \+ (?=\w)", "+"
        )
    )

    .withColumnRenamed("id", "id_filme")
    .select("id_filme", F.col("nome_padronizado").alias("nome_pessoa_empresa"), "tipo_entidade")
    .distinct()
)

# Contagem de linhas pra verificação
print(f"Total de linhas antes de qualquer filtro: {df_todas_entidades.count()}")
print(f"Total de linhas depois de todos os filtros de sujeira: {df_entidades_limpo.count()}")
print(f"Total de linhas depois de remover duplicatas: {df_silver_pessoas_empresas.count()}")

# Checagens de qualidade
# Checando se o ID do filme nao é NULL
dq_check("silver.tb_pessoas_empresas", "id_filme não nulo", df_silver_pessoas_empresas, F.col("id_filme").isNotNull())

# Checando se nome_pessoa_empresa nao é NULL
dq_check("silver.tb_pessoas_empresas", "nome_pessoa_empresa não nulo", df_silver_pessoas_empresas, F.col("nome_pessoa_empresa").isNotNull())

# Checando se tipo_entidade é válido
dq_check("silver.tb_pessoas_empresas", "tipo_entidade válido",
         df_silver_pessoas_empresas,
         F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista", "Produtora"]))

# Mesmo padrçao de gravação com overwrite
df_silver_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

# Print e display de validação
print("Tabela silver.tb_pessoas_empresas gravada com sucesso.")
display(df_silver_pessoas_empresas.orderBy("id_filme").limit(30))

Total de linhas antes de qualquer filtro: 910763
Total de linhas depois de todos os filtros de sujeira: 885133
Total de linhas depois de remover duplicatas: 884390
[PASS] silver.tb_pessoas_empresas | id_filme não nulo | 0/884390 linhas falharam
[PASS] silver.tb_pessoas_empresas | nome_pessoa_empresa não nulo | 0/884390 linhas falharam
[PASS] silver.tb_pessoas_empresas | tipo_entidade válido | 0/884390 linhas falharam
Tabela silver.tb_pessoas_empresas gravada com sucesso.


id_filme,nome_pessoa_empresa,tipo_entidade
14564,Zach Roerig,Ator
14564,Vincent D'onofrio,Ator
14564,Parkes Macdonald Image Nation,Produtora
14564,Laura Slade Wiggins,Ator
14564,Patrick Walker,Ator
14564,Alex Roe,Ator
14564,Kôji Suzuki,Roteirista
14564,Jacob Estes,Roteirista
14564,Bonnie Morgan,Ator
14564,Matilda Lutz,Ator


In [0]:
# PADRONIZANDO NOMES E GRAVANDO silver.tb_avaliacoes_usuarios

# Mapeamento de colunas definido no documento:
#   id -> id_filme | nome -> nome_usuario | nota -> nota_usuario | comentario -> comentario_usuario
df_silver_avaliacoes_usuarios = (
    df_reviews_limpo
    .select(
        F.col("id").alias("id_filme"),
        F.col("nome").alias("nome_usuario"),
        F.col("nota_validada").alias("nota_usuario"),
        F.col("comentario_tratado").alias("comentario_usuario"),
    )
)

# Checagens de qualidade
# ATENÇÃO: aqui NÃO uso dq_check_unique por id_filme -- diferente das outras tabelas, o grão aqui é
# "1 linha por avaliação", e um mesmo filme pode (e deve) ter várias avaliações de usuários diferentes.
dq_check("silver.tb_avaliacoes_usuarios", "id_filme não nulo", df_silver_avaliacoes_usuarios,
         F.col("id_filme").isNotNull())
dq_check("silver.tb_avaliacoes_usuarios", "nota_usuario dentro do intervalo 0-10", df_silver_avaliacoes_usuarios,
         F.col("nota_usuario").isNull() | ((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10)))
dq_check("silver.tb_avaliacoes_usuarios", "comentario_usuario preenchido", df_silver_avaliacoes_usuarios,
         F.col("comentario_usuario").isNotNull())

# Gravando em overwrite
df_silver_avaliacoes_usuarios.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

print("Tabela silver.tb_avaliacoes_usuarios gravada com sucesso.")
display(df_silver_avaliacoes_usuarios.limit(20))

[PASS] silver.tb_avaliacoes_usuarios | id_filme não nulo | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | nota_usuario dentro do intervalo 0-10 | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | comentario_usuario preenchido | 0/32412 linhas falharam
Tabela silver.tb_avaliacoes_usuarios gravada com sucesso.


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
1152100,Isabela Pinto 509,7.6,Legal! Uma boa opção para o fim de semana.
846452,Sérgio Freitas,6.1,Assisti até o final mas não me marcou.
410937,Carolina Lopes 276,2.1,Horrível! Perda de tempo.
760653,Maria Barbosa 699,3.5,Sem comentário
1155082,Carolina Almeida 653,3.2,"Fraco, não recomendo."
923930,Carlos Monteiro 465,9.0,Excepcional! História envolvente e atuações impecáveis.
661409,Matheus Gomes 340,9.8,Perfeito! Um dos melhores filmes que já vi.
447163,Natália Freitas 402,9.1,"Obra-prima do cinema, simplesmente espetacular."
464175,Mônica Cavalcanti,7.7,Filme interessante com boas atuações.


### Correções

In [0]:
# TIPAGEM E TRATAMENTO DE duracao_minutos (descoberto ao validar a camada Gold)

# Três problemas encontrados nessa coluna:

# 1. TIPAGEM: o documento exige que "a tipagem deve estar correta" e a dim_movies especifica
#    duracao_minutos INT, mas a coluna estava como STRING -- ela só tinha sido renomeada de "runtime", sem conversão

# 2. COLUMN SHIFT: a coluna contém texto fora de contexto (ex: " Conor McGregor"), o que fazia
#    qualquer comparação numérica quebrar. Aplico conversão SEGURA: o que não for inteiro válido
#    vira NULL sem interromper o pipeline (mesma estratégia usada nas métricas de engajamento)

# 3. ZEROS: 0 minutos não é uma duração real, é como a origem marca "não informado" -- mesmo
#    raciocínio já aplicado a orçamento e receita. Vira NULL.

# Conversão segura: só vira número se for inteiro válido, senão NULL
duracao_numerica = F.when(
    F.trim(F.col("duracao_minutos")).rlike(r"^\d+$"),
    F.trim(F.col("duracao_minutos")).cast("int")
).otherwise(None)

df_info_filmes_final = df_sinopse_corrigida.withColumn(
    "duracao_minutos",
    F.when(duracao_numerica > 0, duracao_numerica).otherwise(None)
)

# Gravando a tabela já com a duração tipada e tratada
df_info_filmes_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

print("Tabela silver.tb_info_filmes REGRAVADA com duracao_minutos tipada e tratada.")

# Validação
df_check = spark.table(f"{silver_schema}.tb_info_filmes")
df_check.select("duracao_minutos").printSchema()
print(f"Total de filmes: {df_check.count()}")
print(f"duracao_minutos NULL (texto inválido ou zero): {df_check.filter(F.col('duracao_minutos').isNull()).count()}")
print(f"duracao_minutos válida: {df_check.filter(F.col('duracao_minutos').isNotNull()).count()}")
display(df_check.select("id_filme", "titulo", "duracao_minutos").limit(20))